In [2]:
import pyarrow.feather as feather
import pandas as pd
import talib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load the dataset
dataset_file_path = Path('/allah/freqtrade/user_data/data/binance/futures/ETH_USDT_USDT-3m-futures.feather')
crypto_data_df = feather.read_feather(dataset_file_path)

# Calculate Triple Exponential Moving Average (TEMA) on closing prices
tema_period = 50
crypto_data_df['tema'] = talib.TEMA(crypto_data_df['close'], timeperiod=tema_period)

# Establish the trend direction
trend_conditions = [
    crypto_data_df['tema'] > crypto_data_df['tema'].shift(1),
    crypto_data_df['tema'] < crypto_data_df['tema'].shift(1)
]
trend_choices = ['UP', 'DOWN']
crypto_data_df['trend'] = np.select(trend_conditions, trend_choices, default='STABLE')

# Identify trend change points
crypto_data_df['is_trend_change'] = crypto_data_df['trend'] != crypto_data_df['trend'].shift(1)
crypto_data_df['is_trend_change_point'] = crypto_data_df['is_trend_change'] & (crypto_data_df['trend'] != 'STABLE')

# Create a separate DataFrame for change points to avoid SettingWithCopyWarning
trend_change_points_df = crypto_data_df[crypto_data_df['is_trend_change_point']].copy()
trend_change_points_df['trend_duration'] = trend_change_points_df.index.to_series().diff().fillna(0)

# Format data for plotting
trend_change_points_df = trend_change_points_df.round({'tema': 2, 'volume': 2})
if 'date' in trend_change_points_df.columns:
    trend_change_points_df['date'] = trend_change_points_df['date'].dt.strftime('%Y-%m-%d %H:%M')


test
test2
